## Relion (or Relion + CryoSPARC) CTF Plotting

This notebook can be used to visualize micrograph CTFs from ReLiOn in a similar graph format to those from CryoSPARC. It can also be used to overlay CTF graphs from processing in ReLiOn and CryoSPARC to compare between the methods. 

Minimum inputs:
1. Change CTFFIND_JOB_PATH and OUTDIR to reflect your directories

Other inputs:
2. If you have >4 CTF jobs to compare, you will have to increase the number of colors provided in the set below
3. If you want to compare CryoSPARC jobs to ReLiOn jobs, change CS_DIR to the directory containing the .csv file(s) that contain CTF information from CryoSPARC. To obtain this file, build a Curate Exposures job, and then click "Download table CSV" (at the top of the micrographs table, next to the Filter dropdown). For ease of labeling graphs, I recommend changing the name of the CSV to something understandable (ie CryoSPARC.csv).

In [ ]:
# %pip install starfile # <-- unhash if you need to install

import os
import numpy as np
import pandas as pd
import starfile
import matplotlib.pyplot as plt

In [ ]:
# If also comparing to CryoSPARC data
import glob

In [ ]:
### CHANGE ME ###

CTFFIND_JOB_PATH = "/your/relion/directory/CtfFind/job0XX"
OUTDIR = "/directory/to/write/output/images"

# Set overall colors; change or add more if desired
colors = ('lightsteelblue', 'mediumturquoise', 'wheat', 'lightgray')

os.chdir(CTFFIND_JOB_PATH)

In [ ]:
# Read the CSV file into a DataFrame
df_starfile = starfile.read('micrographs_ctf.star')
df_relion = df_starfile['micrographs']

df_relion.head()

In [ ]:
# Plot a scatterplot
plt.scatter(df_relion.index, df_relion['rlnCtfMaxResolution'], s=1, alpha=0.5, color=colors[0], label='Relion')
plt.axhline(df_relion['rlnCtfMaxResolution'].mean(), color=plt1_color, linestyle='dashed')

# plt.title('Title')
plt.xlabel('Micrograph Index')
plt.ylabel('CTF Fit')
plt.ylim(2,6) # <-- change if needed
plt.legend()

# png_name = os.path.join(OUTDIR, "relion_CTF_scatter.png")
# plt.savefig(png_name)

plt.show()

In [ ]:
# Compare to CryoSPARC data on same axes

In [ ]:
### CHANGE ME ###

# Folder containing downloaded CryoSPARC CSV files (cannot contain other CSVs as well)
CS_DIR = "/path/to/csv/files"

In [ ]:
# Find all CSV files in the folder
csv_files = glob.glob(os.path.join(CS_DIR, "*.csv"))

df_cryosparc = {}  # dictionary to hold dataframes

for fpath in csv_files:
    # Use filename without extension as the dataframe name
    name = os.path.splitext(os.path.basename(fpath))[0]
    
    # Read CSV; adjust parameters if needed
    df = pd.read_csv(fpath)
    
    # Ensure the index column is numeric and usable as x-axis
    # If "Index" is a column, set it as index or keep as column
    if "Index" in df.columns:
        df["Index"] = pd.to_numeric(df["Index"], errors="coerce")
    
    df_cryosparc[name] = df

# Expose them as variables in the notebook's global namespace
globals().update(df_cryosparc)

df_cryosparc.keys()  # show the loaded dataframe names

In [ ]:
for key in df_cryosparc:
    print(key)

In [ ]:
# Scatterplot

count = 1
for key in df_cryosparc:
    plt.scatter(df_cryosparc[key].index, df_cryosparc[key]['CTF Fit'], s=1, alpha=0.5, color=colors[count], label=key)
    plt.axhline(df_cryosparc[key]['CTF Fit'].mean(), color=colors[count], linestyle='dashed')
    count = count + 1
    
plt.scatter(df_relion.index, df_relion['rlnCtfMaxResolution'], s=1, color=colors[0], label='Relion')
plt.axhline(df_relion['rlnCtfMaxResolution'].mean(), color=plt1_color, linestyle='dashed')

# plt.title('Title')
plt.xlabel('Micrograph Index')
plt.ylabel('CTF Fit')
plt.ylim(2,8)
plt.legend(markerscale=5, loc='upper right')

# png_name = os.path.join(OUTDIR, "CTF_scatter_all.png")
# plt.savefig(png_name)

plt.show()